# OracleSubscription: Marketplace-Planauswahl und Bicep


In [ ]:
TENANT_ID="8fd3e0b6-0302-4084-a31f-1dfc5bbd03f5"
AZURE_SUBSCRIPTION_ID="4ce5b5cc-e869-451f-98fc-f50448849ee0"
API_VERSION="2025-09-01"
ORACLE_URL="https://management.azure.com/subscriptions/$AZURE_SUBSCRIPTION_ID/providers/Oracle.Database/oracleSubscriptions"
REQUIRED_PROVIDERS='["Oracle.Database","Microsoft.BareMetal","Microsoft.Network","Microsoft.Compute"]'

# Planliste und Auswahlzelle fuellen die drei Planwerte automatisch.
SELECTED_PLAN_ID="oracle_database_at_azure_pay_as_you_go"
BICEP_FILE="./oracle-subscription.bicep"
DEPLOYMENT_LOCATION="francecentral"
PLAN_PUBLISHER=""
PLAN_PRODUCT=""
PLAN_NAME=""

# Nur fuer die jeweilige schreibende Zelle setzen.
REGISTER_CONFIRMATION=""

printf "Tenant: %s\nAzure-Subscription: %s\n" "$TENANT_ID" "$AZURE_SUBSCRIPTION_ID"


## 1. Voraussetzungen pruefen

Dieses Notebook erstellt ausschliesslich eine **neue OracleSubscription** in der
oben eingetragenen Azure-Subscription. Es enthaelt keinen Tenant-Umzug,
keine Hilfsfunktionen, keinen Vorschau-Modus und keine `if/else`-Verzweigungen.

Die Codezellen verwenden den **Bash-Jupyter-Kernel** aus der Projekt-`.venv`;
in VS Code muss die Zellensprache **Shell Script** sein.

Login separat im Terminal ausfuehren, weil eine Notebook-Zelle keine
interaktive Anmeldung/MFA verarbeiten soll:

```bash
az login --tenant 8fd3e0b6-0302-4084-a31f-1dfc5bbd03f5 --use-device-code
```

Danach die Konfigurationszelle oben ausfuehren. `az`, `jq` und eine gueltige
Azure-Anmeldung werden vorausgesetzt. Alle Abfragen sind **live**.
Nur die optional markierte Registrierungszelle veraendert Provider-Registrierungen.
Die Bicep-Zelle zeigt ausschliesslich what-if und kauft nichts. Nicht ungeprueft **Run All** verwenden.

Fehler stoppen die jeweilige Zelle. Strikte Fehlerbehandlung steht in einer
Subshell, damit der Bash-Kernel und Variablen bei einem Fehler erhalten bleiben.


### 1.1 Werkzeuge und Tenant-Zuordnung

Die Subscription muss im angegebenen Tenant liegen und den Status `Enabled`
haben. Die zweite Abfrage benoetigt einen gueltigen Azure-Zugriff; bei
Authentifizierungsfehlern die Anmeldung im Terminal erneuern.


In [ ]:
(
  set -euo pipefail
  az version
  jq --version
  account=$(az account show --subscription "$AZURE_SUBSCRIPTION_ID" -o json)
  jq '{id,tenantId,state}' <<<"$account"
  jq -e --arg tenant "$TENANT_ID" --arg subscription "$AZURE_SUBSCRIPTION_ID" \
    '.tenantId == $tenant and .id == $subscription and .state == "Enabled"' \
    <<<"$account" >/dev/null
  az rest --method get --url \
    "https://management.azure.com/subscriptions/$AZURE_SUBSCRIPTION_ID?api-version=2022-12-01" \
    --query '{subscriptionId:subscriptionId,state:state}' -o json
)


### 1.2 Provider-Registrierung pruefen

Alle vier Provider muessen `Registered` sein. Die Zelle zeigt die vorhandenen
Zustaende und schlaegt fehl, wenn ein erforderlicher Provider fehlt oder noch
nicht registriert ist. In diesem Fall optional 1.3 ausfuehren und 1.2 wiederholen.


In [ ]:
(
  set -euo pipefail
  providers=$(az provider list --subscription "$AZURE_SUBSCRIPTION_ID" -o json)
  jq -r --argjson required "$REQUIRED_PROVIDERS" \
    '.[] | select(.namespace as $n | $required | index($n)) |
      [.namespace,.registrationState] | @tsv' <<<"$providers"
  jq -e --argjson required "$REQUIRED_PROVIDERS" \
    '[.[] | select(.namespace as $n | $required | index($n))] |
      length == 4 and all(.[]; .registrationState == "Registered")' \
    <<<"$providers" >/dev/null
)


### 1.3 Optional: Provider registrieren

**Ueberspringen, wenn 1.2 bereits erfolgreich war.** Fuer diese Zelle in der
Konfiguration `REGISTER_CONFIRMATION="REGISTER"` setzen und die
Konfigurationszelle erneut ausfuehren. Die Registrierung erzeugt keine Datenbank
und kauft kein Angebot. `--wait` wartet auf die Provider-Registrierung.
Anschliessend 1.2 erneut ausfuehren.


In [ ]:
(
  set -euo pipefail
  test "$REGISTER_CONFIRMATION" = "REGISTER"
  az provider register --subscription "$AZURE_SUBSCRIPTION_ID" --namespace Oracle.Database --wait
  az provider register --subscription "$AZURE_SUBSCRIPTION_ID" --namespace Microsoft.BareMetal --wait
  az provider register --subscription "$AZURE_SUBSCRIPTION_ID" --namespace Microsoft.Network --wait
  az provider register --subscription "$AZURE_SUBSCRIPTION_ID" --namespace Microsoft.Compute --wait
)


### 1.4 Pruefen, dass noch keine OracleSubscription existiert

Die folgende Abfrage listet OracleSubscriptions in der Zielsubscription.
Eine vorhandene Ressource wird nicht ueberschrieben. Auch unvollstaendige oder
fehlerhafte API-Antworten bestehen die Pruefung nicht.


In [ ]:
(
  set -euo pipefail
  subscriptions=$(az rest --method get --url "$ORACLE_URL?api-version=$API_VERSION")
  jq '.value | map({id,plan,provisioningState:.properties.provisioningState})' <<<"$subscriptions"
  jq -e '.value | type == "array" and length == 0' <<<"$subscriptions" >/dev/null
  jq -e '(.nextLink // "") == ""' <<<"$subscriptions" >/dev/null
)


### 1.5 Kaufberechtigung und Angebot pruefen

Vor der Erstellung muss ein berechtigter Kaeufer Preise, Laufzeit, Bedingungen
und Marketplace-/Billing-Berechtigungen bestaetigen. Beim Private Offer ist die
Annahme des konkreten Angebots erforderlich; die Auswahl unten ist fuer den
oeffentlichen PAYG-Plan vorgesehen.

Die folgende Zelle zeigt **alle im oeffentlichen Katalog zurueckgegebenen Plaene**
dieses Oracle-Angebots, einschliesslich Private-/Stop-Sell-/Hidden-Markierungen.
Individuelle Kunden-Private-Offers und tenantbezogene Kaufberechtigungen sind
damit nicht nachgewiesen. Eine Planliste ist keine Preisfreigabe.

Die Katalogabfrage ist oeffentlich und kauft nichts. Fuer eure Anforderung ist
`SELECTED_PLAN_ID="oracle_database_at_azure_pay_as_you_go"` vorkonfiguriert.
Keine Anzeigenamen oder kurzen SKU-Kennungen wie `003J` als ARM-Planname verwenden.


In [ ]:
CATALOG_JSON=$(curl --fail --silent --show-error --connect-timeout 15 --max-time 60 "https://catalogapi.azure.com/offers/oracle.oracle_database_at_azure?api-version=2018-08-01-beta") &&
jq -e '.publisherId == "oracle" and .offerId == "oracle_database_at_azure" and
  (.plans | type == "array" and length > 0)' <<<"$CATALOG_JSON" >/dev/null &&
printf "PLAN_ID\tANZEIGENAME\tPRIVAT\tSTOP_SELL\tHIDDEN\n" &&
jq -r '.plans[] | [.planId,.displayName]' <<<"$CATALOG_JSON"


### 1.6 PAYG-Plan aus dem Katalog auswaehlen

Diese Zelle ersetzt die bisherige reine Plananzeige. Sie uebernimmt Publisher,
Produkt und die ausgewaehlte Plan-ID aus der vorherigen Katalogantwort. Genau
ein oeffentlicher, nicht ausgeblendeter und nicht eingestellter Treffer muss
vorhanden sein. Die Variablen bleiben fuer das Bicep-Deployment im Bash-Kernel
erhalten. Diese Zelle erstellt noch keine OracleSubscription.


In [ ]:
PLAN_PUBLISHER=""
PLAN_PRODUCT=""
PLAN_NAME=""

PLAN_PUBLISHER=$(jq -er '.publisherId' <<<"$CATALOG_JSON") &&
PLAN_PRODUCT=$(jq -er '.offerId' <<<"$CATALOG_JSON") &&
PLAN_NAME=$(jq -er --arg selected "$SELECTED_PLAN_ID" \
  '[.plans[] | select(.planId == $selected and .isPrivate == false and
    .isStopSell == false and .isHidden == false)] |
    select(length == 1) | .[0].planId' <<<"$CATALOG_JSON") &&
jq -n --arg publisher "$PLAN_PUBLISHER" --arg product "$PLAN_PRODUCT" \
  --arg name "$PLAN_NAME" '{publisher:$publisher,product:$product,name:$name}'


## 2. OracleSubscription mit Bicep pruefen und erstellen

Bicep erstellt die **Ressource**; Planliste und Planauswahl bleiben Bash/jq.
Das [Bicep-Template](oracle-subscription.bicep) liegt neben dem Notebook und
verwendet `targetScope='subscription'`.

Die folgende Kompilierungszelle ist lokal und loest keinen Kauf aus.
Falls `az bicep` noch nicht installiert ist, im Terminal `az bicep install`
ausfuehren. `BICEP_FILE` muss auf die Datei neben dem Notebook zeigen; startet
der Kernel in einem anderen Verzeichnis, den Pfad in der Konfiguration anpassen.


In [ ]:

  az bicep build --file ./oracle-subscription.bicep

### Bicep what-if: nur geplante Aenderungen anzeigen

Diese Zelle verwendet feste Subscription-, Standort-, Template- und PAYG-Planwerte.
Sie braucht weder Kaufbestaetigung noch Planvariablen aus vorherigen Zellen.
Die Voraussetzungen werden in Abschnitt 1 geprueft, nicht nochmals hier.

`az deployment sub what-if` zeigt die geplanten Aenderungen und erstellt keine
OracleSubscription. Der ungefilterte Bericht bleibt sichtbar. What-if ersetzt
keine Preis-/Vertragsfreigabe und garantiert keine spaetere Kaufberechtigung.

Der Template-Pfad ist relativ zum Notebook-Verzeichnis. Falls der Kernel woanders
startet, zuerst in das Verzeichnis `infra/notebooks` wechseln.
France Central ist hier der Speicherort der Deployment-Metadaten, keine DB-Region.

### Deployment nach der Vorschau

Zelle 19 zeigt nur die geplanten Aenderungen. Zelle 20 fuehrt mit
`az deployment sub create` das echte Deployment mit denselben festen Werten aus.

> [!WARNING]
> Zelle 20 erstellt die OracleSubscription und kann eine kostenpflichtige
> Marketplace-Transaktion ausloesen. Vorher What-if und Vertragsbedingungen pruefen.


In [ ]:
az deployment sub what-if \
  --subscription 4ce5b5cc-e869-451f-98fc-f50448849ee0 \
  --name odaa-oracle-subscription \
  --location francecentral \
  --template-file ./oracle-subscription.bicep \
  --parameters \
    planPublisher=oracle \
    planProduct=oracle_database_at_azure \
    planName=oracle_database_at_azure_pay_as_you_go


In [ ]:
az deployment sub create \
  --subscription 4ce5b5cc-e869-451f-98fc-f50448849ee0 \
  --name odaa-oracle-subscription \
  --location francecentral \
  --template-file ./oracle-subscription.bicep \
  --parameters \
    planPublisher=oracle \
    planProduct=oracle_database_at_azure \
    planName=oracle_database_at_azure_pay_as_you_go


## 3. Erstellte OracleSubscription pruefen

Die folgende GET-Zelle prueft die Ressource nach dem Deployment aus Zelle 20.
What-if allein erstellt nichts; ohne Deployment ist `ResourceNotFound` zu erwarten.

`provisioningState="Succeeded"` bestaetigt die Erstellung der Ressource.
`cloudAccountState="Pending"` bedeutet, dass die OCI-Kontoaktivierung noch aussteht.
Nach interaktivem Oracle-Onboarding steht der API-Zustand auf `Available`.
Das Notebook fuehrt keine OCI-Aktivierung aus.


In [ ]:
(
  set -euo pipefail
  subscription=$(az rest --method get --url "$ORACLE_URL/default?api-version=$API_VERSION")
  jq '{id,plan,properties:(.properties | {
    provisioningState,saasSubscriptionId,cloudAccountState,cloudAccountId
  })}' <<<"$subscription"
  jq -e --arg id "/subscriptions/$AZURE_SUBSCRIPTION_ID/providers/Oracle.Database/oracleSubscriptions/default" \
    '(.id | ascii_downcase) == ($id | ascii_downcase) and
      .properties.provisioningState == "Succeeded"' <<<"$subscription" >/dev/null
  printf "OracleSubscription erfolgreich erstellt. OCI-Aktivierung separat abschliessen.\n"
)


### Dokumentierte APIs

* [OracleSubscriptions auflisten](https://learn.microsoft.com/rest/api/oracle/oracle-subscriptions/list-by-subscription?view=rest-oracle-2025-09-01)
* [OracleSubscription erstellen](https://learn.microsoft.com/rest/api/oracle/oracle-subscriptions/create-or-update?view=rest-oracle-2025-09-01)
* [OracleSubscription pruefen](https://learn.microsoft.com/rest/api/oracle/oracle-subscriptions/get?view=rest-oracle-2025-09-01)
* [Oracle-Angebot kaufen](https://docs.oracle.com/en-us/iaas/Content/database-at-azure/onboard-purchase.htm)

* [Bicep: OracleSubscription auf Subscription-Scope](https://learn.microsoft.com/azure/templates/oracle.database/2025-09-01/oraclesubscriptions)
* [Oeffentlicher Oracle-Marketplace-Katalog](https://catalogapi.azure.com/offers/oracle.oracle_database_at_azure?api-version=2018-08-01-beta)
